In [1]:
#utils
import pandas as pd
import numpy as np
def indices(lst, item):
    return [i for i, x in enumerate(lst) if x == item]

def validate_dataframes(human_data, mouse_data, human_covar):
     # Check if the provided data is pandas DataFrame and not empty
     for df in [human_data, mouse_data, human_covar]:
         if df is not None:
             if not isinstance(df, pd.DataFrame) or df.empty:
                 raise ValueError("Data must be a non-empty pandas DataFrame.")
                 
def validate_hvf_and_pcs(hvf, n_pcs, mouse_data):
     if not (isinstance(hvf, int) or isinstance(hvf, list)):
         raise ValueError("If you want to use hvf, please pass a positive integer that is less than the total number of features in the dataset or a self-defined hvf list. Otherwise set hvf to zero.")
     if isinstance(hvf, int):
         if hvf < 0 or hvf > mouse_data.shape[1]:
             raise ValueError("If you want to use hvf, please pass a positive integer that is less than the total number of features in the dataset or a self-defined hvf list. Otherwise set hvf to zero.")

     if not (isinstance(n_pcs, int) and n_pcs > 0) and n_pcs != 'max':
         raise ValueError("n_pcs must be a positive integer or 'max'.")
 
def validate_classes(human_classes):
     if not isinstance(human_classes, (pd.Series, pd.DataFrame, np.ndarray)):
         raise ValueError("human_classes should be a pandas Series, DataFrame or a numpy array. If human_classes is a list, please use numpy.array(human_classes) instead")
     else:
         if isinstance(human_classes, np.ndarray):
             # convert numpy array to pandas Series
             human_classes = pd.Series(human_classes.flatten(), name='target')
     return human_classes    

#tools
import time
from Bio import Entrez
import pandas as pd
import numpy as np
import warnings
import gseapy as gp

def gsea(results, metric='db', species='Human', top_n_pcs=5, pcs=None, top_n_features=25, save_path=None, **kwargs):
    
    """
    Run GSEA on the top genes of the principal components (PCs) that best differentiate two organism2 classes.
    
    Parameters
    ----------
    results : dict
        A dictionary containing results from previous computations. Should contain keys 
        'predictivity_summary' and 'organism2_transComps'.
    
    metric : str, optional
        Metric for selecting top principal components. Valid options include 'db', 'coefs',
        'indiv acc', 'ch', 'mean diff', and 'tp'. Default is 'db'.
    
    species : str
        Specifies the species for which the GSEA should be performed. Must be one of 
        'Human', 'Mouse', 'Yeast', 'Fly', 'Fish', or 'Worm'.
    
    top_n_pcs : int, optional
        Consider the top N principal components. Default is 5.
    
    pcs : int or list of int, optional
        Specific PCs to be used. When specified, these PCs are used in place of top PCs.
    
    top_n_features : int
        Top N features on both sides of each PC for GSEA.
    
    save_path : str, optional
        Directory where the GSEA table CSV file should be saved. If None, GSEA results 
        won't be saved locally. Default is None.
    
    kwargs : dict, optional
        Additional keyword arguments to be passed to the function.
    
    Returns
    -------
    GSEA_up_result, GSEA_down_result : DataFrame
        The GSEA result tables for upregulated and downregulated genes, respectively.
    
    Notes
    -----
    The function analyzes the principal components (PCs) that best differentiate two 
    classes of organism2 based on a specified metric.
    """
    warnings.filterwarnings("ignore")
    if species not in ['Human', 'Mouse', 'Yeast', 'Fly', 'Fish', 'Worm']:
        raise ValueError("Invalid species. Please choose from ['Human', 'Mouse', 'Yeast', 'Fly', 'Fish', 'Worm'].")

    if metric == 'db':
        sorted_df = results['predictivity_summary'].sort_values(by='davies_bouldin_score', ascending=True)
    elif metric == 'coefs':
        if sum(results['predictivity_summary']['coefs'] != 0) >= top_n_pcs:
            results['predictivity_summary']['coefs_abs'] = abs(results['predictivity_summary']['coefs'])
            sorted_df = results['predictivity_summary'].sort_values(by='coefs_abs', ascending=False)
        else:
            raise ValueError("You don't have enough pcs that have a non-zero coef. Decrease your top_n_pcs.")
    elif metric == 'indiv acc':
        sorted_df = results['predictivity_summary'].sort_values(by='individual_predictivity', ascending=False)
    elif metric == 'ch':
        sorted_df = results['predictivity_summary'].sort_values(by='Calinski-Harabasz Score', ascending=False)
    elif metric == 'mean diff':
        sorted_df = results['predictivity_summary'].sort_values(by='Difference of Means', ascending=False)
    elif metric == 'tp':
        sorted_df = results['predictivity_summary'].sort_values(by='t-test p-value', ascending=True)
    else:
        raise ValueError("Invalid metric. Please choose one from 'coefs', 'indiv acc', 'db','ch','mean diff','tp'.")

    top_ids = sorted_df.index[0:top_n_pcs]
    if pcs is not None:
        top_ids = list(pcs)

    # Fetching the features of organism1 based on sorting, then retrieving the corresponding features from organism2
    top_pc_features_org1 = sorted_df['sorted_organism1_features'].loc[top_ids]
    organism1_features = results['organism1_features']
    organism2_features = results['organism2_features']

    # Retrieve the corresponding organism2 features based on the indices of the features in organism1
    up_features_org2 = []
    down_features_org2 = []
    for features in top_pc_features_org1:
        indices_up = [organism1_features.get_loc(feature) for feature in features[-top_n_features:]]
        indices_down = [organism1_features.get_loc(feature) for feature in features[:top_n_features]]
        up_features_org2.append([organism2_features[idx] for idx in indices_up])
        down_features_org2.append([organism2_features[idx] for idx in indices_down])

    GSEA_up_result = {}
    for i in range(0, len(top_ids)):
        GOBP_up = gp.enrichr(gene_list=up_features_org2[i],
                             gene_sets=['GO_Biological_Process_2021'],
                             organism=species)
        GSEA_up_result[int(top_ids[i])] = GOBP_up.results
        
        GOBP_up.results['-log10_p_adj'] = -np.log10(GOBP_up.results['Adjusted P-value'])
        GOBP_up.results.index = GOBP_up.results['Term']
        
        if save_path is not None:
            GOBP_up.results.to_csv(save_path+'GSEA_PC'+str(int(top_ids[i])+1)+'_UP.csv')
            
    GSEA_down_result = {}
    for i in range(0, len(top_ids)):  
        GOBP_down = gp.enrichr(gene_list=down_features_org2[i],
                               gene_sets=['GO_Biological_Process_2021'],
                               organism=species)
        GSEA_down_result[int(top_ids[i])] = GOBP_down.results
        
        GOBP_down.results['-log10_p_adj'] = -np.log10(GOBP_down.results['Adjusted P-value'])
        GOBP_down.results.index = GOBP_down.results['Term']
                
        if save_path is not None:
            GOBP_down.results.to_csv(save_path+'GSEA_PC'+str(int(top_ids[i])+1)+'_DOWN.csv')
    
    return GSEA_up_result, GSEA_down_result

def gsea_filtered(results, metric='db', species='Human', top_n_pcs=5, pcs=None, top_n_features=25, save_path=None, **kwargs):
    
    """
    Run GSEA on the top genes of the principal components (PCs) that best differentiate two organism2 classes.
    
    Parameters
    ----------
    results : dict
        A dictionary containing results from previous computations. Should contain keys 
        'predictivity_summary' and 'organism2_transComps'.
    
    metric : str, optional
        Metric for selecting top principal components. Valid options include 'db', 'coefs',
        'indiv acc', 'ch', 'mean diff', and 'tp'. Default is 'db'.
    
    species : str
        Specifies the species for which the GSEA should be performed. Must be one of 
        'Human', 'Mouse', 'Yeast', 'Fly', 'Fish', or 'Worm'.
    
    top_n_pcs : int, optional
        Consider the top N principal components. Default is 5.
    
    pcs : int or list of int, optional
        Specific PCs to be used. When specified, these PCs are used in place of top PCs.
    
    top_n_features : int
        Top N features on both sides of each PC for GSEA.
    
    save_path : str, optional
        Directory where the GSEA table CSV file should be saved. If None, GSEA results 
        won't be saved locally. Default is None.
    
    kwargs : dict, optional
        Additional keyword arguments to be passed to the function.
    
    Returns
    -------
    GSEA_up_result, GSEA_down_result : DataFrame
        The GSEA result tables for upregulated and downregulated genes, respectively.
    
    Notes
    -----
    The function analyzes the principal components (PCs) that best differentiate two 
    classes of organism2 based on a specified metric.
    """
    warnings.filterwarnings("ignore")
    if species not in ['Human', 'Mouse', 'Yeast', 'Fly', 'Fish', 'Worm']:
        raise ValueError("Invalid species. Please choose from ['Human', 'Mouse', 'Yeast', 'Fly', 'Fish', 'Worm'].")

    if metric == 'db':
        sorted_df = results['predictivity_summary'].sort_values(by='davies_bouldin_score', ascending=True)
    elif metric == 'coefs':
        if sum(results['predictivity_summary']['coefs'] != 0) >= top_n_pcs:
            results['predictivity_summary']['coefs_abs'] = abs(results['predictivity_summary']['coefs'])
            sorted_df = results['predictivity_summary'].sort_values(by='coefs_abs', ascending=False)
        else:
            raise ValueError("You don't have enough pcs that have a non-zero coef. Decrease your top_n_pcs.")
    elif metric == 'indiv acc':
        sorted_df = results['predictivity_summary'].sort_values(by='individual_predictivity', ascending=False)
    elif metric == 'ch':
        sorted_df = results['predictivity_summary'].sort_values(by='Calinski-Harabasz Score', ascending=False)
    elif metric == 'mean diff':
        sorted_df = results['predictivity_summary'].sort_values(by='Difference of Means', ascending=False)
    elif metric == 'tp':
        sorted_df = results['predictivity_summary'].sort_values(by='t-test p-value', ascending=True)
    else:
        raise ValueError("Invalid metric. Please choose one from 'coefs', 'indiv acc', 'db','ch','mean diff','tp'.")

    top_ids = sorted_df.index[0:top_n_pcs]
    if pcs is not None:
        top_ids = list(pcs)

    # Fetching the features of organism1 based on sorting, then retrieving the corresponding features from organism2
    top_pc_features_org1 = sorted_df['sorted_organism1_features'].loc[top_ids]
    organism1_features = results['organism1_features']
    organism2_features = results['organism2_features']

    # Retrieve the corresponding organism2 features based on the indices of the features in organism1
    up_features_org2 = []
    down_features_org2 = []
    for features in top_pc_features_org1:
        indices_up = [organism1_features.get_loc(feature) for feature in features[-top_n_features:]]
        indices_down = [organism1_features.get_loc(feature) for feature in features[:top_n_features]]
        up_features_org2.append([organism2_features[idx] for idx in indices_up])
        down_features_org2.append([organism2_features[idx] for idx in indices_down])

    GSEA_up_result_filtered = {}
    for i in range(0, len(top_ids)):
        GOBP_up = gp.enrichr(gene_list=up_features_org2[i],
                             gene_sets=['GO_Biological_Process_2021'],
                             organism=species)
        GSEA_up_result[int(top_ids[i])] = GOBP_up.results
        top_id = top_ids[i]

         # Access the DataFrame for the current top_id
        df = GSEA_up_result[int(top_id)]
        
        # Calculate the overlap percentage
        df['Gene Set Size'] = df['Overlap'].apply(lambda x: int(x.split('/')[1]))  # Get the total gene set size
        df['Overlap Count'] = df['Overlap'].apply(lambda x: int(x.split('/')[0]))  # Get the overlap count
        df['Overlap Percentage'] = (df['Overlap Count'] / df['Gene Set Size']) * 100
        
        # Filter for pathways with at least 10% overlap
        filtered_df = df[df['Overlap Percentage'] >= 5]
        
        # Add -log10(Adjusted P-value) for easier significance interpretation
        filtered_df['-log10_p_adj'] = -np.log10(filtered_df['Adjusted P-value'])
        
        # Set the 'Term' column as the index
        filtered_df.index = filtered_df['Term']
        
        # Save the filtered results back to the GSEA_up_result dictionary
        GSEA_up_result_filtered[int(top_id)] = filtered_df
        
        if save_path is not None:
            GOBP_up.results.to_csv(save_path+'GSEA__Filtered_PC'+str(int(top_ids[i])+1)+'_UP.csv')

    GSEA_down_result_filtered = {}
    for i in range(0, len(top_ids)):  
        GOBP_down = gp.enrichr(gene_list=down_features_org2[i],
                               gene_sets=['GO_Biological_Process_2021'],
                               organism=species)
        GSEA_down_result[int(top_ids[i])] = GOBP_down.results
        top_id = top_ids[i]

         # Access the DataFrame for the current top_id
        df = GSEA_down_result[int(top_id)]
        
        # Calculate the overlap percentage
        df['Gene Set Size'] = df['Overlap'].apply(lambda x: int(x.split('/')[1]))  # Get the total gene set size
        df['Overlap Count'] = df['Overlap'].apply(lambda x: int(x.split('/')[0]))  # Get the overlap count
        df['Overlap Percentage'] = (df['Overlap Count'] / df['Gene Set Size']) * 100
        
        # Filter for pathways with at least 10% overlap
        filtered_df = df[df['Overlap Percentage'] >= 10]
        
        # Add -log10(Adjusted P-value) for easier significance interpretation
        filtered_df['-log10_p_adj'] = -np.log10(filtered_df['Adjusted P-value'])
        
        # Set the 'Term' column as the index
        filtered_df.index = filtered_df['Term']
        
        # Save the filtered results back to the GSEA_down_result dictionary
        GSEA_down_result_filtered[int(top_id)] = filtered_df
                
        if save_path is not None:
            GOBP_down.results.to_csv(save_path+'GSEA_Filtered_PC'+str(int(top_ids[i])+1)+'_DOWN.csv')
    
    return GSEA_up_result_filtered, GSEA_down_result_filtered



def highly_variable_features(df, n_top_features):
    """
    Identifies the most variable features in the dataset.
    
    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame containing feature expression data. Columns should represent
        feature names, and rows should correspond to individual samples.
        
    n_top_features : int
        The number of top variable features to return.
    
    Returns
    -------
    list
        A list containing the names of the most variable features.
    """


    mean = df.mean(axis=0)
    variance = df.var(axis=0)
    df_fluctuation = pd.DataFrame({'mean': mean, 'variance': variance})
    df_fluctuation["residuals"] = np.log10(df_fluctuation["variance"]) - np.log10(df_fluctuation["mean"])
    
    # Fitting a linear regression model
    slope, intercept = np.polyfit(np.log10(df_fluctuation["mean"]), np.log10(df_fluctuation["variance"]), 1)
    df_fluctuation["fitted"] = intercept + slope * np.log10(df_fluctuation["mean"])
    
    # Subtract fitted values from the observed values
    df_fluctuation["dispersion"] = df_fluctuation["residuals"] - df_fluctuation["fitted"]
    
    # Selecting the most variable features
    df_fluctuation = df_fluctuation.sort_values(by='dispersion', ascending=False)
    hvf = df_fluctuation.head(n_top_features).index.tolist()
    
    return hvf

#transcompR
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, make_scorer, davies_bouldin_score, calinski_harabasz_score, r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate
import warnings
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from bayes_opt import BayesianOptimization
from scipy.stats import ttest_ind
import statsmodels.api as sm

def TransCompR(organism1_data, organism2_data, organism2_classes, organism2_covar = None,non_cat_covar = [],
               n_pcs = 50, hvf = 0, use_interaction=False, penalty = 'l1', 
               l1_ratio = None, use_BO = False, BO_iter = 20, cv = 5,
               random_state = None, **kwargs):
    """
    Perform analysis and regression on input data from two organisms.
    
    Parameters
    ----------
    organism1_data : DataFrame
        A pandas DataFrame of shape (n_organism1, n_features) with matched feature columns 
        based on homology with organism2_data. Pre-processed data is expected.
    
    organism2_data : DataFrame
        A pandas DataFrame of shape (n_organism2, n_features) with matched feature columns 
        based on homology with organism1_data. Pre-processed data is expected.
    
    organism2_classes : Series or array-like
        Phenotypes for regression for organism2.
    
    organism2_covar : DataFrame, optional
        A DataFrame of shape (organism2, categorical_info). Categorical information for organism2.
        Dummy variables will be automatically generated, so they are not accepted. Default is None.
    
    n_pcs : int or str, optional
        Number of principal components to use. If 'max', maximum number of PCs will be used.
        Default is 50.
    
    hvf : int or list, optional
        Defines highly variable features. If 0, all features are used. If it's an integer, top 'hvf' features with 
        the largest residual in a mean-variance relationship regression will be used. If it's a list of feature symbols 
        for organism1, then that list will be used. Default is 0.
    
    use_interaction : bool, optional
        Whether to include interactions between TranscComps and covariates. Default is False.
    
    penalty : {'l1', 'l2', 'elasticnet', None}, optional
        The penalty type used for logistic regression. Default is 'l1'.
    
    l1_ratio : float, optional
        Parameter used only when penalty is 'elasticnet'. It should be between 0 and 1. Default is None.
    
    use_BO : bool, optional
        If True, use Bayesian Optimizer for selecting C in logistic regression. Default is False.
    
    BO_iter : int, optional
        Number of iterations for the Bayesian optimizer. Used only if use_BO is True. Default is 20.
    
    cv : int, optional
        Number of folds for cross-validation. Used only if use_BO is True. Default is 5.
    
    random_state : int or RandomState instance, optional
        Random state seed used for shuffling data when solver is one of {'sag', 'saga', 'liblinear'}. Default is None.
    
    Returns
    -------
    results : dict
        A dictionary with results and metrics from the regression analysis. Contains keys:
        - 'organism1_hvf': List of calculated highly variable features in organism1 data if 'hvf' is not None.
        - 'model': Logistic regression model for organism2 classes.
        - 'organism1_loadings': DataFrame (n_pcs, n_features) with loadings of organism1 features.
        ... [Include other keys similarly formatted]
    
    Notes
    -----
    Ensure that both organism data inputs are pre-processed and have matching features based on homology.
    """
    warnings.filterwarnings("ignore")
    
    n_features = organism1_data.shape[1]
    
    # Z-score normalize feature-matched organism1 and organism2 data matrices
    scaler = StandardScaler()
    organism2_Zdata = scaler.fit_transform(organism2_data.values)
    organism1_Zdata = scaler.fit_transform(organism1_data.values)
    
    # Train organism1 Data PCA Model
    if hvf == 0:
        hvf = organism1_data.columns.tolist()
    elif isinstance(hvf, int) and hvf > 0 and hvf <= organism1_data.shape[1]:
        hvf = highly_variable_features(organism1_data,hvf)   
    elif isinstance(hvf, list):
        hvf = hvf
    else:
        raise ValueError("If you want to use hvf, please pass an positive integer that is less than the total number of features in the dataset. Otherwise set False.")
    keep_id = np.where([i in hvf for i in organism1_data.columns.tolist()])
    organism1_data = organism1_data.iloc[:,keep_id[0]]
    organism1_Zdata = organism1_Zdata[:,keep_id[0]]
    organism2_data = organism2_data.iloc[:,keep_id[0]]
    organism2_Zdata = organism2_Zdata[:,keep_id[0]]

    
    if isinstance(n_pcs,str) and n_pcs == 'max':
        n_pcs = np.min(organism1_data.shape)
        print('Max n_pcs used. Set n_pcs to '+str(n_pcs))
    elif isinstance(n_pcs,int) and n_pcs > np.min(organism1_data.shape): 
        n_pcs = np.min(organism1_data.shape)
        print('n_pcs exceeded the maximum. Set n_pcs to '+str(n_pcs))
    elif isinstance(n_pcs,int) and n_pcs <= 0: 
        n_pcs = np.min(organism1_data.shape)
        print('n_pcs cannot be non-negative. Set n_pcs to '+str(n_pcs))
    elif isinstance(n_pcs,int) == False:
        raise ValueError("Invalid n_pcs argument.")
   
    pca = PCA(n_components = n_pcs, random_state= random_state)
    organism1_scores = pca.fit_transform(organism1_Zdata)
    organism1_loadings = pd.DataFrame(pca.components_)
    organism1_loadings.columns = organism1_data.columns
    organism1_explained = pca.explained_variance_ratio_
    
    # Project organism2 data Into organism1 PCA
    organism2_transComps = scaler.fit_transform(np.dot(organism2_Zdata, organism1_loadings.T))
    organism2_transComps = pd.DataFrame(organism2_transComps, index = organism2_data.index)

    organism2_transComps_noZ = np.dot(organism2_Zdata, organism1_loadings.T)
    organism2_transComps_noZ = pd.DataFrame(organism2_transComps_noZ, index = organism2_data.index)     

    transCompR_table = pd.concat([organism2_transComps, pd.DataFrame(organism2_classes)], axis=1)
    transCompR_table_noZ = pd.concat([organism2_transComps_noZ, pd.DataFrame(organism2_classes)], axis=1)
    
    # Identify organism1 PCs predictive of organism2 classes
    # Assuming last column in table is the target variable for regression
    X = transCompR_table.iloc[:,:-1]
    X_noZ = transCompR_table_noZ.iloc[:,:-1]
    y = transCompR_table.iloc[:,-1]
    
    if organism2_covar is not None and isinstance(organism2_covar, pd.DataFrame):
       #Now we deal with covariates
        prefix = dict()
        noncat_covar = organism2_covar[non_cat_covar]
        noncat_covar_noZ = noncat_covar.copy()
        for i in noncat_covar.columns:
            continuous_covar = []
            for j in noncat_covar[i]:
                try:
                    continuous_covar.append(float(j))
                except:
                    continuous_covar.append(np.nan)
            noncat_covar[i] = scaler.fit_transform(np.array(continuous_covar).reshape(-1,1))
            noncat_covar_noZ[i] = np.array(continuous_covar)
            
        cat_covar = organism2_covar.iloc[:,~organism2_covar.columns.isin(non_cat_covar)]
        for i in cat_covar.columns:
            cat_covar[i] = cat_covar[i].astype('category')
            prefix[i] = 'dummy_'+i
        add_dummies = pd.get_dummies(cat_covar, prefix = prefix)
        add_dummies_noZ = pd.concat([add_dummies, noncat_covar_noZ],axis = 1)
        add_dummies = pd.concat([add_dummies, noncat_covar],axis = 1)

    #Adding interaction terms
        if use_interaction:
            for trans_col in organism2_transComps.columns:
                for dummy_col in add_dummies.columns:
                    interaction_col_name = f"interaction_{trans_col}_{dummy_col}"
                    X[interaction_col_name] = organism2_transComps[trans_col] * add_dummies[dummy_col]
                    X_noZ[interaction_col_name] = organism2_transComps_noZ[trans_col] * add_dummies_noZ[dummy_col]
            
        X = pd.concat([X, add_dummies],axis = 1)
        X_noZ = pd.concat([X_noZ, add_dummies_noZ],axis = 1)
        
    X.columns = X.columns.astype(str)
    new_tcr_table = pd.concat([X,y],axis = 1)
    new_tcr_table = new_tcr_table.dropna()
    
    X = new_tcr_table.iloc[:,:-1]
    y = new_tcr_table.iloc[:,-1]

    X_noZ.columns = X_noZ.columns.astype(str)
    new_tcr_table = pd.concat([X_noZ,y],axis = 1)
    new_tcr_table = new_tcr_table.dropna()
    X_noZ = new_tcr_table.iloc[:,:-1]
    
    # Split data into training and testing
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/cv, random_state=random_state)

    if penalty == 'l1':
        solver_name = 'liblinear'
    elif penalty == 'l2':
        solver_name = 'lbfgs'
    elif penalty == 'elasticnet':
        solver_name = 'saga'        
    elif penalty == 'None':
        solver_name = 'lbfgs'
    else:
        print("Invalid penalty argument. Use no penalty.")

   
    # Refit the best model to the entire dataset
    transCompR_mdl = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
    y_predict = transCompR_mdl.predict(X_test)
    y_predict = pd.Series(y_predict)
    print('y predict:', type(y_predict))
    y_predict.index = y_test.index

    accuracy = accuracy_score(y_test, y_predict)
    precision = precision_score(y_test, y_predict)
    recall = recall_score(y_test, y_predict)
    mse = mean_squared_error(y_test, y_predict)
    mae = mean_absolute_error(y_test, y_predict)
    R2score = r2_score(y_test, y_predict)
    auc_score = roc_auc_score(y_test, y_predict)
    
    scores = pd.DataFrame([auc_score, recall, precision, accuracy, R2score, mse, mae]).T
    scores.columns = ['AUC', 'Recall', 'Precision', 'Accuracy', 'R^2 score', 'MSE', 'MAE',]

    predictions = pd.DataFrame([y_test, y_predict]).T
    predictions.columns = ['y test', 'y predict']

    #5-fold cross-validation
    transCompR_cross = RandomForestClassifier(n_estimators=100, random_state=42)
    cross_val = cross_validate(transCompR_cross, X, y, scoring = ['roc_auc','accuracy', 'f1','recall','precision'], cv=5)
    
    top_organism1 = [organism1_data.columns[np.argsort(organism1_loadings.iloc[i,:])].tolist() for i in range(0,n_pcs)] #ascending
    top_organism1_loadings = [organism1_loadings.values[i,np.argsort(organism1_loadings.iloc[i,:])] for i in range(0,n_pcs)]
    top_coef_top_features = pd.DataFrame({'PCs': [str(i) for i in range(1,n_pcs+1)],
                                   'sorted_organism1_features': top_organism1,
                                   'sorted_organism1_features_loadings': top_organism1_loadings,
                                   'organism1_explained': np.var(organism1_scores, axis = 0)/n_features,
                                   'organism2_explained': np.var(organism2_transComps_noZ, axis = 0)/n_features})
    
    if organism2_covar is not None and isinstance(organism2_covar, pd.DataFrame):
        indiv_accuracy = np.zeros(n_pcs)
        for i in range(0,n_pcs):
            indiv_X = X.iloc[:,i]
            if use_interaction:
                for dummy_col in add_dummies.columns:
                    interaction_col_name = f"interaction_{trans_col}_{dummy_col}"
                    indiv_X[interaction_col_name] = X.iloc[:,i] * add_dummies[dummy_col]
                indiv_X = pd.concat([indiv_X, add_dummies],axis = 1)
            else:
                indiv_X = pd.concat([indiv_X, add_dummies],axis = 1)
            
            indiv_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(indiv_X.loc[X_train.index,:], y_train)
            indiv_accuracy[i] = accuracy_score(y_test, indiv_model.predict(indiv_X.loc[X_test.index,:]))
        top_coef_top_features['individual_predictivity'] = indiv_accuracy
          
    db_score = [davies_bouldin_score(X_noZ.iloc[:,i].values.reshape(-1,1),y) for i in range(0,n_pcs)]
    top_coef_top_features['davies_bouldin_score'] = db_score
    
    unique_categories = list(set(y))
    if len(unique_categories) != 2:
        raise ValueError("y should have exactly 2 unique categories for this calculation")
    
    category_1 = unique_categories[0]
    category_2 = unique_categories[1]
    
    # Calculate Calinski and Harabasz Score for each PC
    ch_scores = [calinski_harabasz_score(X_noZ.iloc[:,i].values.reshape(-1,1), y) for i in range(0, n_pcs)]
    top_coef_top_features['calinski_harabasz_score'] = ch_scores
    
    # Calculate Difference of Mean between two categories for each PC
    differences = [X_noZ[y == category_1].iloc[:,i].mean() - X_noZ[y == category_2].iloc[:,i].mean() for i in range(0, n_pcs)]
    top_coef_top_features['difference_of_means'] = differences
    
    # Calculate t-test p-value for difference of means between two categories for each PC
    p_values = []
    for i in range(0, n_pcs):
        category_1_data = X_noZ[y == category_1].iloc[:,i]
        category_2_data = X_noZ[y == category_2].iloc[:,i]
        t_stat, p_value = ttest_ind(category_1_data, category_2_data)
        p_values.append(p_value)
    top_coef_top_features['t_test_p_value'] = p_values

        
    all_regression_coeffs = pd.DataFrame()
  #  all_regression_coeffs['coefs'] = transCompR_mdl.coef_[0]
    all_regression_coeffs.index = X.columns.tolist()
    results = {
        'organism1_features': organism1_data.columns,
        'organism2_features': organism2_data.columns,
        'organism1_hvf' : hvf,
        'model': transCompR_mdl,
        'organism1_loadings': organism1_loadings,
        'organism1_scores': organism1_scores,
        'organism2_transComps': organism2_transComps_noZ,
        'organism2_classes': organism2_classes,
        'X': X,
        'X_noZ': X_noZ,
        'y': y,
        'training':[X_train,y_train],
        'testing': [X_test, y_test],
        'regression_terms': X.columns.tolist(), # Add the regression terms
        'all_regression_coeffs': all_regression_coeffs, # Store all regression coefficients
        'cross_validation_metrics': scores,
        'predictivity_summary': top_coef_top_features,
        'predictions': predictions,
        'cross': cross_val
    }
    return results

#plot
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from adjustText import adjust_text
from sklearn.neighbors import KernelDensity
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from adjustText import adjust_text
from sklearn.metrics import calinski_harabasz_score
from scipy.stats import ttest_ind


def loadings(results, pcs, organism1_name='organism1', top_n_features=20, save_path=None, 
             return_figures=False, figsize=(8,8), fontsize=20, fontweight='bold',
             textsize=8, textcolor='red', textweight='bold', stripdotsize=3, 
             highlightcolor='orange', jitter=0.3, scattersize=5):
    """
    Visualize loadings of features on each principal component for a given organism.
    
    Parameters
    ----------
    results : dict
        Results dictionary with a key such as '<organism>_loadings' that contains a DataFrame of feature loadings.
        
    pcs : list or int
        A list or an integer indicating which principal components to plot.
    
    top_n_features : int
        Number of top features to be highlighted on each PC.
    
    organism1_name : str, optional
        Name of organism1. Defaults to 'organism1'. Note: PCA is done on organism 1, so technically the loadings
        are an attribute of the organism1's genes.
    
    save_path : str, optional
        Path to save the figures. If None, figures will not be saved. Default is None.
    
    return_figures : bool, optional
        If True, returns the matplotlib.figure.Figure instances. Default is False.
    
    figsize : tuple, optional
        Size of the figure. Default is (8,8).
    
    fontsize : int, optional
        Font size for plot labels and title. Default is 30.
    
    fontweight : str, optional
        Font weight for plot labels and title. Default is 'bold'.
    
    textsize : int, optional
        Text size for annotations. Default is 8.
    
    textcolor : str, optional
        Text color for annotations. Default is 'red'.
    
    textweight : str, optional
        Text weight for annotations. Default is 'bold'.
    
    stripdotsize : int, optional
        Size of the dots in the strip plot. Default is 3.
    
    highlightcolor : str, optional
        Color to highlight the outliers. Default is 'orange'.
    
    jitter : float, optional
        Amount of jitter (only along the categorical axis) to apply. This can be useful when many points overlap, making it easier to see the distribution. Default is 0.3.
    
    scattersize : int, optional
        Size of the dots in the scatter plot. Default is 5.
    
    Returns
    -------
    figures : list or None
        List of matplotlib.figure.Figure instances if return_figures is True, else None.
    
    Raises
    ------
    ValueError
        If pcs is not an integer or a list of 2 integers.
    """

    
    plt.rcParams.update({'font.size': 14, 'font.weight': 'bold', 'axes.linewidth': 2})
    organism_loadings = results['organism1_loadings']

    figures = []
    
    for i in pcs:
        if isinstance(i,int):
            fig, ax = plt.subplots(1,1,figsize=figsize)
            data = organism_loadings.iloc[i,:].T.sort_values()

            outliers = pd.concat([data.head(int(np.ceil(top_n_features/2))),data.tail(int(np.floor(top_n_features/2)))])
            insiders = data[int(np.ceil(top_n_features/2)):-int(np.floor(top_n_features/2))]

            path_collection = sns.stripplot(data=outliers, ax=ax, color=highlightcolor, size=stripdotsize*2, jitter=jitter).collections[0]
            coords = path_collection.get_offsets()
            xcoord= [coords[i][0] for i in range(0,len(outliers))]
            ycoord= [coords[i][1] for i in range(0,len(outliers))]

            texts = []
            for j, txt in enumerate(outliers.index):
                texts.append(ax.text(xcoord[j], ycoord[j], txt, size=textsize, color=textcolor, weight=textweight))
        
            sns.boxplot(data=data, ax=ax, fliersize=0, width=jitter*2)
            sns.stripplot(data=insiders, ax=ax, color=(0.3, 0.3, 0.3, 0.6), size=stripdotsize, alpha=0.3, jitter=jitter, zorder=1)

            ax.set_xlabel(f'{organism1_name}-derived PC ' + str(i + 1), size=fontsize, weight=fontweight)
            ax.set_ylabel('Loadings', size=fontsize, weight=fontweight)
            ax.set_xticks([0])
            ax.set_xticklabels('')
            adjust_text(texts)
            figures.append(fig)
            
        elif isinstance(i, list) and all([isinstance(j,int) for j in i]):
            if len(i) == 2:
                data = organism_loadings.iloc[i,:].T
                data['abs_sum'] = abs(data[i[0]]) + abs(data[i[1]])
                plot_entries = data.sort_values(by='abs_sum', ascending=False).head(top_n_features)
                labels = plot_entries.index.tolist()
                plot_entries = plot_entries.reset_index()
                
                fig, ax = plt.subplots(figsize=figsize)
    
                # move the left and bottom spines to x=0 and y=0, respectively.
                ax.spines['left'].set_position('center')
                ax.spines['bottom'].set_position('center')
    
                # remove the right and top spines
                ax.spines['right'].set_color('none')
                ax.spines['top'].set_color('none')
    
                # remove the ticks on the top and right axes
                ax.xaxis.set_ticks_position('bottom')
                ax.yaxis.set_ticks_position('left')
                
                ax.scatter(data.iloc[:,0], data.iloc[:,1], s=scattersize*0.5, alpha=0.2, color='grey')
                ax.scatter(plot_entries.iloc[:,1], plot_entries.iloc[:,2], s=scattersize*2, color=highlightcolor)

                texts = []
                for j, txt in enumerate(labels):
                    texts.append(ax.text(plot_entries.iloc[j,1], plot_entries.iloc[j,2], txt, size=textsize, color=textcolor, weight=textweight))
                adjust_text(texts)
                
                ax.text(-0.05, 0.5, organism1_name+'-derived PC'+str(i[0]+1), va='center', ha='right', size=fontsize, transform=ax.transAxes, weight=fontweight,rotation='vertical')
                ax.text(0.5, 0, organism1_name+'-derived PC'+str(i[1]+1), va='top', ha='center', size=fontsize, transform=ax.transAxes, weight=fontweight)
                figures.append(fig)
            else:
                raise ValueError("Only support 2D plot.")
        else:
            raise ValueError("Invalid pcs argument.")
    
        if save_path is not None:
            fig.savefig(save_path+'Loadings_PC'+str(np.array(i)+1)+'.png')

    if return_figures:
        return figures

def var_explained(results, organism1_name='organism1', organism2_name='organism2',
                  n_pcs=20,  facecolor=['#00AFB9','#F07167'], save_path=None, 
                  return_figures=False, **kwargs):
    """
    Visualize the total variance explained in data from two organisms by the principal components (PCs) of organism1.
    
    Parameters
    ----------
    results : dict
        A dictionary containing results with a key 'predictivity_summary'. This key should have a sub-dictionary with 
        'organism2_explained' and 'organism1_explained' keys. These represent the percentage variance explained by 
        organism1's PCs in organism2 and organism1 data, respectively.
    
    organism1_name : str, optional
        Name of the first organism for visualization purposes. Defaults to 'organism1'.
    
    organism2_name : str, optional
        Name of the second organism for visualization purposes. Defaults to 'organism2'.
    
    n_pcs : int, optional
        Number of PCs to include in the visualization. Defaults to 20.
    
    facecolor : list, optional
        Colors for the two classes in the plot. Defaults to ['#D2493A','#28548f'].
    
    save_path : str, optional
        Path where the figure should be saved. If None, the figure won't be saved. Defaults to None.
    
    return_figures : bool, optional
        If True, returns the matplotlib.figure.Figure instance. Defaults to False.
    
    **kwargs : dict, optional
        Additional keyword arguments to be passed to the matplotlib.pyplot.bar.
    
    Returns
    -------
    matplotlib.figure.Figure or None
        A matplotlib.figure.Figure instance if return_figures is set to True, otherwise None.
    """

    
    organism1_explained = results['predictivity_summary']['organism1_explained']*100
    organism2_explained = results['predictivity_summary']['organism2_explained']*100
    
    if n_pcs > len(organism1_explained):
        print(str(len(organism1_explained)) + ' PCs at maximum. Set n_pcs to ' + str(len(organism1_explained)))
        n_pcs = len(organism1_explained)

    fig, axs = plt.subplots(1, 2, figsize=(n_pcs, 6))
    
    axs[0].bar([str(i) for i in range(1, n_pcs+1)], organism1_explained[0:n_pcs], color=facecolor[0], **kwargs)
    axs[0].set_title(f'Explained {organism1_name} Variance ('+ str(np.round(np.sum(organism1_explained[0:n_pcs]),decimals = 2))+'% explained by top '+str(n_pcs)+' PCs)', fontsize = 14)
    axs[0].set_xlabel(f'{organism1_name}-derived PC')
    axs[0].set_ylabel('Percentage (%)')
    
    axs[1].bar([str(i) for i in range(1, n_pcs+1)], organism2_explained[0:n_pcs], color=facecolor[1], **kwargs)
    axs[1].set_title(f'Explained {organism2_name} Variance ('+ str(np.round(np.sum(organism2_explained[0:n_pcs]),decimals = 2))+'% explained by top '+str(n_pcs)+' PCs)', fontsize=14)
    axs[1].set_xlabel(f'{organism1_name}-derived PC')
    axs[1].set_ylabel('Percentage (%)')
    
    if save_path is not None:
        fig.savefig(save_path + 'var_explained.png', bbox_inches='tight')
    
    if return_figures:
        return fig
    
def gsea_bar(results, GSEA_up_result, GSEA_down_result, n_go=10, facecolor=['#D2493A','#28548f'],save_path=None, return_figures=False, **kwargs):
    
    """
    Visualizes the GSEA (Gene Set Enrichment Analysis) result.
    
    Parameters
    ----------
    GSEA_up_result : dict
        A dictionary where keys are the top PCs and values are DataFrames of the GSEA results for upregulated genes.
    
    GSEA_down_result : dict
        A dictionary where keys are the top PCs and values are DataFrames of the GSEA results for downregulated genes.
    
    n_go : int, optional
        The number of gene ontology terms to consider. Defaults to 10.
    
    facecolor : list, optional
        Colors for the two classes in the plot. Defaults to ['#D2493A','#28548f'].
    
    save_path : str, optional
        Path where the generated figures should be saved. If None, figures won't be saved. Defaults to None.
    
    return_figures : bool, optional
        If True, returns the generated figures. Defaults to False.
    
    kwargs : dict, optional
        Additional keyword arguments to pass to the bar plot function.
    
    Returns
    -------
    list or None
        A list of generated figures if return_figures is set to True. Otherwise, returns None.
    """

    warnings.filterwarnings("ignore")
    figures_up = []
    figures_down = []
    
    top_ids = list(GSEA_up_result.keys())
    
    scores = results['organism2_transComps'].loc[:, top_ids]
    label = results['organism2_classes']
    label = label.iloc[:-1]
    label = label.reset_index()
    scores_class1 = scores[label == list(set(label))[0]]
    scores_class2 = scores[label == list(set(label))[1]]

    direction = []
    for i in np.array([int(i) for i in top_ids]):
        if np.mean(scores_class1[i]) > np.mean(scores_class2[i]):
            direction.append('(increases ' + str(list(set(label))[0]) + ' group)')
        elif np.mean(scores_class1[i]) < np.mean(scores_class2[i]):
            direction.append('(decreases ' + str(list(set(label))[0]) + ' group)')
        else:
            direction.append('(groups has no polarization on this PC)')
    
    for i in range(0, len(top_ids)):
        fig, axs = plt.subplots()
        GSEA_up_result[int(top_ids[i])][['-log10_p_adj']][0:n_go].sort_values(by = '-log10_p_adj', ascending = True).plot.barh(xlabel = '-log10 adjusted P-value',color=facecolor[0],legend =None,title = 'GSEA PC'+str(int(top_ids[i])+1)+' Upregulated Pathways',ax =axs, **kwargs)
        figures_up.append(fig)
        
        if save_path is not None:
            fig.savefig(save_path+'GSEA_PC'+str(int(top_ids[i])+1)+'_UP.png',bbox_inches='tight')
            
    for i in range(0, len(top_ids)):
        fig, axs = plt.subplots()    
        GSEA_down_result[int(top_ids[i])][['-log10_p_adj']][0:n_go].sort_values(by = '-log10_p_adj', ascending = True).plot.barh(xlabel = '-log10 adjusted P-value',color=facecolor[1],legend =None,title = 'GSEA PC'+str(int(top_ids[i])+1)+' Downregulated Pathways',ax = axs,**kwargs)
        figures_down.append(fig)
        
        if save_path is not None:
            fig.savefig(save_path+'GSEA_PC'+str(int(top_ids[i])+1)+'_DOWN.png',bbox_inches='tight')

    if return_figures:
            return figures_up, figures_down

def partition(results, organism2_name='organism2', metric='db', top_n_pcs=5,
              pcs=None, covar = None, save_path=None, return_figures=False,
              fontsize=12, fontweight='bold',
              facecolor=["#219EBC", "#FB8500"], cmap ='YlGnBu',  **kwargs):
    """
    Visualizes the Kernel Density Estimation (KDE) of organism2 samples based on the top Principal Components (PCs) 
    that best separate two organism2 classes, as determined by a specified metric.
    
    Parameters
    ----------
    results : dict
        A dictionary from prior computations containing keys 'predictivity_summary' and 'organism2_transComps'.
    
    organism2_name : str, optional
        Name of organism2 for visualization. Defaults to 'organism2'.
    
    metric : str, optional
        Metric for selecting top principal components. Accepts 'db', 'coefs', 'indiv acc', 'ch', 'mean diff', and 'tp'. 
        Defaults to 'db'.
    
    top_n_pcs : int, optional
        The top N principal components under consideration. Defaults to 5.
    
    pcs : int or list of int, optional
        When provided, this specific PC(s) is/are plotted in lieu of the top PCs.
    
    covar : str or list of str, optional
        If provided, the function visualizes the separation of organism2 classes based on the input covariates.
    
    save_path : str, optional
        Directory path where generated figures should be saved. If None, figures are not saved. Defaults to None.
    
    return_figures : bool, optional
        If True, returns the generated figures. Defaults to False.
    
    fontsize : int, optional
        Font size for the plot's labels and title. Defaults to 12.
    
    fontweight : str, optional
        Font weight for the plot's labels and title. Defaults to 'bold'.
    
    facecolor : list, optional
        Colors for the two classes in the plot. Defaults to ["#219EBC", "#FB8500"].
    
    cmap : str or colormap object, optional
        Colormap passed to the heatmap. Utilized if 'covar' contains dummy variables.
    
    kwargs : dict, optional
        Additional keyword arguments for the bar plot function.
    
    Returns
    -------
    list or None
        A list of generated figures if return_figures is True. Otherwise, returns None.
    """
    figures = []
    if covar is not None:
        if not isinstance(covar, list):
            covar = [covar]
        for covar_names in covar:
            scores = results['X_noZ'][covar_names].values
            label = results['y']   # Organism2 classes are considered here
            
            fig, axs = plt.subplots()
            if len(list(set(scores))) ==2:
                df = pd.DataFrame()
                df[covar_names] = scores
                df[organism2_name+' classes'] = np.array(label)
                pivot_table = df.groupby([covar_names, organism2_name+' classes']).size().unstack(fill_value=0)
                sns.heatmap(pivot_table, annot=True, cmap=cmap, cbar_kws={'label': 'Frequency'},ax= axs,**kwargs)
                axs.set_title('Heatmap of classes separation in covariate ' + covar_names, fontsize = 12,fontweight = fontweight)
                figures.append(fig)
                
                if save_path is not None:
                    fig.savefig(save_path+'Partition_covar_'+str(covar_names)+'.png')
            else: 
                scores_class1 = scores[label == list(set(label))[0]]
                scores_class2 = scores[label == list(set(label))[1]]
                scores_plot = np.linspace(np.min(scores), np.max(scores), 100)[:, np.newaxis]
                
                kde1 = KernelDensity(kernel="gaussian", bandwidth=0.75).fit(scores_class1.reshape(-1, 1))
                log_dens1 = kde1.score_samples(scores_plot)
                axs.fill_between(scores_plot[:,0], np.exp(log_dens1), fc=facecolor[0],**kwargs)
                
                kde2 = KernelDensity(kernel="gaussian", bandwidth=0.75).fit(scores_class2.reshape(-1, 1))
                log_dens2 = kde2.score_samples(scores_plot)
                axs.fill_between(scores_plot[:,0], np.exp(log_dens2), fc=facecolor[1],**kwargs)
                axs.legend(list(set(label)))
                
                axs.set_title('Organism2 classes separation on covariate '+ covar_names,weight = fontweight)
    
                axs.set_xlabel(covar_names, size=fontsize, weight=fontweight)
                axs.set_ylabel('Normalized Gaussian KDE',size=fontsize, weight=fontweight)
                figures.append(fig)
        
            if save_path is not None:
                fig.savefig(save_path+'Partition_covar_'+str(covar_names)+'.png')
    else:
        if metric == 'db':
            title_kw = 'Davies-Bouldin Score ='
            sort_kw = 'davies_bouldin_score'
            sorted_df = results['predictivity_summary'].sort_values(by=sort_kw, ascending=True)
            top_ids = sorted_df.index[0:top_n_pcs]
            if pcs is not None:
                if pcs is not None:
                    if isinstance(pcs, int):
                        top_ids = [pcs]
                    else:
                        top_ids = pcs
            metric_values = sorted_df[sort_kw].loc[top_ids]
            scores = results['organism2_transComps'].iloc[:, top_ids]

        elif metric == 'coefs':
            if sum(results['predictivity_summary']['coefs']!=0)>=top_n_pcs:
                title_kw = 'Regression Coefficient ='
                results['predictivity_summary']['coefs_abs'] = abs(results['predictivity_summary']['coefs'])
                sorted_df = results['predictivity_summary'].sort_values(by = 'coefs_abs',ascending = False)
                
                top_ids = sorted_df.index[0:top_n_pcs]
                if pcs is not None:
                    if pcs is not None:
                        if isinstance(pcs, int):
                            top_ids = [pcs]
                        else:
                            top_ids = pcs
                metric_values = sorted_df['coefs'].loc[top_ids]
                scores = results['organism2_transComps'].iloc[:,top_ids]                   
            else:
                raise ValueError("You don't have enough pcs that have a non-zero regression coef. Decrease your top_n_pcs.")
        elif metric == 'indiv acc':
            title_kw = 'Individual Regression accuracy ='
            sorted_df = results['predictivity_summary'].sort_values(by = 'individual_predictivity',ascending = False) 
            top_ids = sorted_df.index[0:top_n_pcs]
            if pcs is not None:
                if pcs is not None:
                    if isinstance(pcs, int):
                        top_ids = [pcs]
                    else:
                        top_ids = pcs
            metric_values = sorted_df['individual_predictivity'].loc[top_ids]
            scores = results['organism2_transComps'].iloc[:,top_ids]
            
        elif metric == 'ch':
            title_kw = 'Calinski-Harabasz Score ='
            sorted_df = results['predictivity_summary'].sort_values(by = 'Calinski-Harabasz Score',ascending = False) 
            top_ids = sorted_df.index[0:top_n_pcs]
            if pcs is not None:
                if isinstance(pcs, int):
                    top_ids = [pcs]
                else:
                    top_ids = pcs
            metric_values = sorted_df['Calinski-Harabasz Score'].loc[top_ids]
            scores = results['organism2_transComps'].iloc[:,top_ids]
            
        elif metric == 'mean diff':
            title_kw = 'Difference of Means ='
            sorted_df = results['predictivity_summary'].sort_values(by = 'Difference of Means',ascending = False) 
            top_ids = sorted_df.index[0:top_n_pcs]
            if pcs is not None:
                if pcs is not None:
                    if isinstance(pcs, int):
                        top_ids = [pcs]
                    else:
                        top_ids = pcs
            metric_values = sorted_df['Difference of Means'].loc[top_ids]
            scores = results['organism2_transComps'].iloc[:,top_ids]
            
        elif metric == 'tp':
            title_kw = 't-test p-value ='
            sorted_df = results['predictivity_summary'].sort_values(by = 't-test p-value',ascending = True) 
            top_ids = sorted_df.index[0:top_n_pcs]
            if pcs is not None:
                if pcs is not None:
                    if isinstance(pcs, int):
                        top_ids = [pcs]
                    else:
                        top_ids = pcs
            metric_values = sorted_df['t-test p-value'].loc[top_ids]
            scores = results['organism2_transComps'].iloc[:,top_ids]

        else:
            raise ValueError("Invalid metric. Please choose one from 'coefs', 'indiv acc', 'db','ch','mean diff','tp'.")

        label = results['organism2_classes']   # Organism2 classes are considered here
        label = label.iloc[:-1]
        scores_class1 = scores[label == list(set(label))[0]]
        scores_class2 = scores[label == list(set(label))[1]]
        
        for i in range(0, len(top_ids)):
            fig, axs = plt.subplots()
            scores_plot = np.linspace(np.min(scores.iloc[:,i]), np.max(scores.iloc[:,i]), 100)[:, np.newaxis]
            
            kde1 = KernelDensity(kernel="gaussian", bandwidth=0.75).fit(scores_class1.iloc[:,i].values.reshape(-1, 1))
            log_dens1 = kde1.score_samples(scores_plot)
            axs.fill_between(scores_plot[:,0], np.exp(log_dens1), fc=facecolor[0],**kwargs)
            
            kde2 = KernelDensity(kernel="gaussian", bandwidth=0.75).fit(scores_class2.iloc[:,i].values.reshape(-1, 1))
            log_dens2 = kde2.score_samples(scores_plot)
            axs.fill_between(scores_plot[:,0], np.exp(log_dens2), fc=facecolor[1],**kwargs)
            axs.legend(list(set(label)))
            
            axs.set_title(title_kw + '%s' % float('%.3g' % metric_values.iloc[i]),weight = fontweight )

            axs.set_xlabel(organism2_name+' TransComp ' + str(scores.columns[i]+1) + ' Scores', size=fontsize, weight=fontweight)
            axs.set_ylabel('Normalized Gaussian KDE',size=fontsize, weight=fontweight)
            figures.append(fig)
        
            if save_path is not None:
                fig.savefig(save_path+'Partition_TransComp'+str(int(str(top_ids[i]))+1)+'_'+metric+'.png')


    if return_figures:
        return figures

def covariate_pc_interaction(results, organism2_covar, covar_key, pc, is_categorical = True,
                             nbins = 5, facecolor=["#219EBC", "#FB8500"],
                             save_path = None,return_figures=False,**kwargs):
    #single pair of covar_pcs
    #Again, python index starts from 0, if you want pc1, type 0 here
    """
    Visualizes the interaction between covariates and principal components (PCs) using either a box plot for categorical covariates or a bar plot for continuous covariates.
    
    Parameters
    ----------
    results : dict
        Dictionary with information about organisms and their associated PCs.
        - 'organism2_transComps': DataFrame with organisms as indices and PCs as columns.
        - 'organism2_classes': DataFrame or column indicating the class/category of each organism.
    
    organism2_covar : dict
        Dictionary containing covariate information for each organism.
    
    covar_key : str
        Key to access specific covariate information within the organism2_covar dictionary.
    
    pc : int
        Index of the principal component (0-based index, e.g., 0 for PC1).
    
    is_categorical : bool, optional
        If the covariate is categorical or not. Defaults to True. If False, assumes continuous.
    
    nbins : int, optional
        Number of bins for binning continuous covariates. Used only if is_categorical is False. Defaults to 5.
    
    facecolor : list of str, optional
        List of two colors for plotting. First color is for class1 and second for class2. Defaults to ["#219EBC", "#FB8500"].
    
    save_path : str, optional
        If provided, saves the resulting figure to this path. Defaults to None.
    
    return_figures : bool, optional
        If True, returns the generated figure object. Otherwise, displays it. Defaults to False.
    
    Returns
    -------
    matplotlib.figure.Figure or None
        If return_figures is True, returns the generated figure. Otherwise, None.
    
    Examples
    --------
    To visualize the interaction of a categorical covariate with PC1:
    >>> covariate_pc_interaction(results_dict, organism_covar_dict, 'covariate_key', 0)
    
    To visualize the interaction of a continuous covariate with PC1 and save the figure:
    >>> covariate_pc_interaction(results_dict, organism_covar_dict, 'covariate_key', 0, is_categorical=False, save_path='./path/to/save/')
    """

   
    if is_categorical:
        df = pd.DataFrame()
        df.index = results['organism2_transComps'].index
        df['score'] = results['organism2_transComps'].iloc[:,pc].values
        df['covar_cat'] = organism2_covar[covar_key].astype('category')
        df['classes'] = results['organism2_classes']
        df['covar_cat_classes'] = df['covar_cat'].astype(str).map(str) + '_'+df['classes'].astype(str).map(str)
        df = df.sort_values(by = 'covar_cat_classes')
        fig, axs = plt.subplots(figsize=(8, 6))
        sns.boxplot(x='covar_cat_classes', y='score', data=df, ax = axs,**kwargs)
        for label in axs.get_xticklabels():
            label.set_rotation(90)
        axs.set_ylabel('PC '+ str(pc+1)+' Score')
        axs.set_xlabel(covar_key+' + classes')
        if save_path is not None:
            fig.savefig(save_path+organism2_covar+'_PC'+str(pc+1)+'.png',bbox_inches='tight')
        if return_figures:
            return fig
    else:
        print('Assume the covariate is continuous')
        df = pd.DataFrame()
        df.index = results['organism2_transComps'].index
        df['score'] = results['organism2_transComps'].iloc[:,pc].values
        df['classes'] = results['organism2_classes']
        
        continuous_covar = []
        for i in organism2_covar[covar_key]:
            try:
                continuous_covar.append(float(i))
            except:
                continuous_covar.append(np.nan)
        data = np.array(continuous_covar)
        df['continuous_covar'] = data
        df=df.dropna()
        
        bin_edges = np.floor(np.linspace(start=np.nanmin(df['continuous_covar']), stop=np.nanmax(df['continuous_covar']), num=nbins+1))
        # Create bin labels with ranges
        bin_labels = [f"{bin_edges[i]} to {bin_edges[i+1]}" for i in range(nbins)]
        # Use pandas.cut to bin the data
        binned_data = pd.cut(df['continuous_covar'], bins=bin_edges, labels=bin_labels, include_lowest=True)
        # Convert binned_data to an array of strings (bin names)
        group_names = np.array(binned_data.astype(str))
        
        df['covar_cat'] = group_names
        class1_avg_scores = df.loc[df['classes'] == list(set(df['classes']))[0],:].groupby('covar_cat')['score'].mean()
        class2_avg_scores = df.loc[df['classes'] == list(set(df['classes']))[1],:].groupby('covar_cat')['score'].mean()

        fig, axs = plt.subplots(figsize=(6, 6))
        # The width of the bars
        bar_width = 0.5  # bars will now take up half of their designated space
        
        # Plot bars for each class
        axs.bar(class1_avg_scores.index, class1_avg_scores, color=facecolor[0], width=bar_width, edgecolor='black', alpha=0.5, label=list(set(df['classes']))[0], align='edge',**kwargs)
        axs.bar(class2_avg_scores.index, class2_avg_scores, color=facecolor[1], width=bar_width, edgecolor='black', alpha=0.5, label=list(set(df['classes']))[1], align='center',**kwargs)
        
        # Setting the labels, title, and custom x-axis tick labels
        axs.set_ylabel('PC '+ str(pc+1)+' Score')
        axs.set_xlabel(covar_key+' groups')
        axs.set_xticklabels(bin_labels, rotation=90)
        axs.legend()  # display the legend
        
        plt.tight_layout()
        if save_path is not None:
            fig.savefig(save_path+organism2_covar+'_PC'+str(pc+1)+'.png',bbox_inches='tight')
        if return_figures:
            return fig